# 07 — Integrated Retail Pipeline
## Classes 27–29 Together

This is the main real-project lab.

```text
Official files
 ↓
UC managed Volume
 ↓
Reusable Python modules
 ↓
Spark execution
 ↓
DQ gates
 ↓
Managed Delta tables
 ↓
Retail KPIs
```

This is a realistic batch pipeline. Structured Streaming comes later in the syllabus.


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / marker).exists():
        PROJECT_ROOT = str(candidate)
        break

if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
from schemas import (
    CUSTOMERS_SCHEMA, PRODUCTS_SCHEMA, ORDERS_SCHEMA,
    ORDER_ITEMS_SCHEMA, PAYMENTS_SCHEMA,
)
from readers import read_csv
from transforms import (
    clean_customers, clean_orders, clean_payments,
    filter_orders, build_order_item_finance,
    build_order_sales, build_city_kpis,
)
from quality import (
    duplicate_keys, orphan_keys, require_zero_rows,
)


In [0]:
dbutils.widgets.dropdown(
    "pipeline_environment", "DEV", ["DEV", "QA", "PROD"],
    "Pipeline Environment"
)
dbutils.widgets.dropdown(
    "pipeline_order_status", "DELIVERED",
    ["DELIVERED", "SHIPPED", "PLACED", "CANCELLED"],
    "Order Status Filter"
)
dbutils.widgets.text(
    "pipeline_city", "", "Shipping City (optional)"
)

pipeline_environment = dbutils.widgets.get("pipeline_environment").upper()
pipeline_order_status = dbutils.widgets.get("pipeline_order_status").upper()
pipeline_city = dbutils.widgets.get("pipeline_city").strip()


In [0]:
customers_raw_df = read_csv(
    spark, f"{ctx.source_root}/customers.csv", CUSTOMERS_SCHEMA
)
products_raw_df = read_csv(
    spark, f"{ctx.source_root}/products.csv", PRODUCTS_SCHEMA
)
orders_raw_df = read_csv(
    spark, f"{ctx.source_root}/orders.csv", ORDERS_SCHEMA
)
order_items_raw_df = read_csv(
    spark, f"{ctx.source_root}/order_items.csv", ORDER_ITEMS_SCHEMA
)
payments_raw_df = read_csv(
    spark, f"{ctx.source_root}/payments.csv", PAYMENTS_SCHEMA
)

print(
    customers_raw_df.count(),
    products_raw_df.count(),
    orders_raw_df.count(),
    order_items_raw_df.count(),
    payments_raw_df.count(),
)


In [0]:
customers_clean_df = clean_customers(customers_raw_df)
orders_clean_df = clean_orders(orders_raw_df)
payments_clean_df = clean_payments(payments_raw_df)

products_clean_df = products_raw_df
order_items_clean_df = order_items_raw_df


In [0]:
require_zero_rows(
    duplicate_keys(orders_clean_df, "order_id"),
    "duplicate order_id",
)
require_zero_rows(
    orphan_keys(order_items_clean_df, orders_clean_df, "order_id"),
    "orphan order item order_id",
)
require_zero_rows(
    orphan_keys(order_items_clean_df, products_clean_df, "product_id"),
    "orphan product_id",
)
require_zero_rows(
    orphan_keys(orders_clean_df, customers_clean_df, "customer_id"),
    "orphan customer_id",
)

print("DQ gates: PASSED")


In [0]:
order_item_finance_df = build_order_item_finance(
    order_items_clean_df,
    products_clean_df,
)

order_sales_df = build_order_sales(
    orders_clean_df,
    order_item_finance_df,
    customers_clean_df,
)

city_kpis_df = build_city_kpis(order_sales_df)

display(city_kpis_df)


In [0]:
orders_runtime_df = filter_orders(
    orders_clean_df,
    order_status=pipeline_order_status,
    shipping_city=pipeline_city,
)

print("Runtime-filtered rows:", orders_runtime_df.count())
display(orders_runtime_df.limit(25))


In [0]:
outputs = {
    "silver_customers": customers_clean_df,
    "silver_orders": orders_clean_df,
    "gold_order_sales": order_sales_df,
    "gold_city_kpis": city_kpis_df,
}

for table_name, output_df in outputs.items():
    (
        output_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(ctx.table(table_name))
    )
    print("Wrote:", ctx.table(table_name))
